以下を行う。
```
Gaussian distribution
        ↓
   Generative Model
        ↓
digit image
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset

# -----------------------
# setup
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# data: sklearn digits
# -----------------------
digits = load_digits()
x = digits.data.astype("float32") / 16.0   # [N, 64], values in [0,1]

x_tensor = torch.tensor(x)
dataset = TensorDataset(x_tensor)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

data_dim = 64

# -----------------------
# Flow Matching model
# u_theta(x_t, t): R^64 x [0,1] -> R^64
# -----------------------
class FlowModel(nn.Module):
    def __init__(self, data_dim=64, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x_t, t):
        return self.net(torch.cat([x_t, t], dim=1))

model = FlowModel(data_dim=data_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# -----------------------
# training
# -----------------------
n_epochs = 500
losses = []

for epoch in range(n_epochs):
    total_loss = 0.0

    for (x1,) in loader:
        x1 = x1.to(device)

        # x0: Gaussian noise
        x0 = torch.randn_like(x1)

        # random time
        t = torch.rand(x1.shape[0], 1, device=device)

        # linear path
        x_t = (1 - t) * x0 + t * x1

        # target velocity
        v_target = x1 - x0

        # predicted velocity
        v_pred = model(x_t, t)

        loss = ((v_pred - v_target) ** 2).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss / len(loader))

    if epoch % 50 == 0:
        print(f"epoch {epoch:4d} | loss = {losses[-1]:.6f}")

# -----------------------
# sampling
# -----------------------
@torch.no_grad()
def sample_digits(model, n_samples=64, n_steps=100):
    model.eval()

    x = torch.randn(n_samples, data_dim, device=device)
    ts = torch.linspace(0, 1, n_steps + 1, device=device)

    for i in range(n_steps):
        t = ts[i].expand(n_samples, 1)
        dt = ts[i + 1] - ts[i]
        x = x + model(x, t) * dt

    x = torch.clamp(x, 0, 1)
    return x.cpu().numpy()

generated = sample_digits(model, n_samples=64, n_steps=200)

# -----------------------
# plot generated digits
# -----------------------
fig, axes = plt.subplots(8, 8, figsize=(6, 6))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(generated[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

plt.suptitle("Generated digits by Flow Matching without AE")
plt.tight_layout()
plt.show()

# -----------------------
# plot loss
# -----------------------
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Flow Matching training loss")
plt.tight_layout()
plt.show()